In [ ]:
path_to_dataset = "" #FIXME: Add path to dataset here

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

dinov2 = torch.hub.load(
    repo_or_dir="facebookresearch/dinov2", 
    model='dinov2_vits14'
)
dinov2 = dinov2.to(device)
dinov2.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
def extract_features_from_image_crops(csv_path, path_to_dataset, label_column='super_category'):
    """Reads a CSV of bounding boxes, crops images, and extracts DINOv2 features."""
    df = pd.read_csv(csv_path)
    
    features = []
    labels = []
    
    for _, row in df.iterrows():
        img_name = row['image_filename']
        subfolder = img_name[0].upper()
        img_path = os.path.join(path_to_dataset, subfolder, img_name)
        
        if not os.path.exists(img_path):
            print(f"Warning: Image not found at {img_path}. Skipping.")
            continue
            
        try:
            # Load image
            img = Image.open(img_path).convert('RGB')
            img_width, img_height = img.size
            
            # Convert normalized YOLO coordinates to absolute pixel coordinates
            x_center = row['x_center'] * img_width
            y_center = row['y_center'] * img_height
            box_width = row['width'] * img_width
            box_height = row['height'] * img_height
            rotation_angle = row['rotation_angle']
            
            left = x_center - (box_width / 2)
            top = y_center - (box_height / 2)
            right = x_center + (box_width / 2)
            bottom = y_center + (box_height / 2)
            
            # Crop the image to the bounding box
            crop_img = img.crop((left, top, right, bottom))

            # Rotate the image if necessary based on the 'rotation_angle' column
            if rotation_angle == 90:
                crop_img = crop_img.rotate(-90, expand=True)
            elif rotation_angle == 180:
                crop_img = crop_img.rotate(180, expand=True)
            elif rotation_angle == 270:
                crop_img = crop_img.rotate(90, expand=True)

            # Prepare tensor and extract features
            img_tensor = transform(crop_img).unsqueeze(0).to(device)
            
            with torch.no_grad():
                feature_vector = dinov2(img_tensor)
                features.append(feature_vector.cpu().numpy().flatten())
                labels.append(row[label_column])
                
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            
    return np.array(features), np.array(labels)

**Feature extraction**</br>
*Only run these cells if feature vectors could not be downloaded*

In [ ]:
X, y = extract_features_from_image_crops(
    "../Training_data/notch_classification_dataset.csv",
    path_to_dataset,
    label_column='super_category'
)
print(f"Extracted {len(X)} feature vectors.")

In [ ]:
np.savez_compressed('dinov2_features.npz', features=X, labels=y)

**Load features instead of extracting**

In [ ]:
data = np.load('dinov2_features.npz', allow_pickle=True)

X = data['features']
y = data['labels']

print(f"Loaded {len(X)} feature vectors.")

In [ ]:
# Split the dataset into training and validation sets (80% train, 20% validation)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# This generates fake 'noise' vectors until the classes are equal in size
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"Original train shape: {np.unique(y_train, return_counts=True)}")
print(f"Balanced train shape: {np.unique(y_train_balanced, return_counts=True)}")

**Training notch-noise SVM**

In [ ]:
notch_noise_classifier = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
notch_noise_classifier.fit(X_train_balanced, y_train_balanced)

# Evaluate the model
y_pred = notch_noise_classifier.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
#  Get probabilities for the test set
probabilities = notch_noise_classifier.predict_proba(X_test)
notch_idx = np.where(notch_noise_classifier.classes_ == 'notch')[0][0]
notch_probs = probabilities[:, notch_idx]

# Automatically find the best threshold
# We want the highest threshold that still gives us 0 False Negatives (missed notches)
best_threshold = 0.0
for thresh in np.arange(0.0, 1.0, 0.01):
    test_preds = np.where(notch_probs >= thresh, 'notch', 'noise')
    
    # Calculate how many actual notches we missed
    missed_notches = np.sum((y_test == 'notch') & (test_preds == 'noise'))
    
    if missed_notches == 0:
        best_threshold = thresh
    else:
        break # As soon as we miss 1 notch, stop. The previous threshold was the best.

print(f"\nOptimal Threshold to catch 100% of notches: {best_threshold:.2f}")

# 6. Evaluate with the optimal threshold
final_preds = np.where(notch_probs >= best_threshold, 'notch', 'noise')

print("\nFinal Confusion Matrix:")
print(confusion_matrix(y_test, final_preds, labels=['noise', 'notch']))

print("\nFinal Classification Report:")
print(classification_report(y_test, final_preds))